# 04 - Évaluation et export

Deux choses, dans l'ordre : évaluer le modèle Keras entraîné sur le test, une fois pour toutes (accuracy, matrice de confusion, F1 macro), puis exporter ce modèle figé en TensorFlow Lite (float32 et int8), en vérifiant que les deux versions restent proches du modèle Keras d'origine.

In [ ]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

DATA_DIR = Path("../data/raw/UCI HAR Dataset")
MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")

## Recharger le modèle figé, la normalisation et le test

In [ ]:
with open(MODELS_DIR / "normalization.json") as f:
    normalization = json.load(f)
with open(MODELS_DIR / "labels.json") as f:
    labels_map = json.load(f)

CHANNELS = normalization["channels"]
mean = np.array(normalization["mean"])
std = np.array(normalization["std"])
validation_subjects = normalization["validation_subjects"]
label_names = [labels_map[str(i)] for i in range(len(labels_map))]

def load_ids(path):
    return pd.read_csv(path, header=None, names=["value"]).squeeze("columns")

def load_signal(split, signal_name):
    path = DATA_DIR / split / "Inertial Signals" / f"{signal_name}_{split}.txt"
    return np.loadtxt(path)

def load_windows(split):
    signals = [load_signal(split, name) for name in CHANNELS]
    return np.stack(signals, axis=-1)

def normalize(X):
    return (X - mean) / std

X_train_full = load_windows("train")
subject_train_full = load_ids(DATA_DIR / "train" / "subject_train.txt")
fit_mask = ~subject_train_full.isin(validation_subjects).to_numpy()
X_fit = normalize(X_train_full[fit_mask]).astype("float32")

X_test = normalize(load_windows("test")).astype("float32")
y_test = (load_ids(DATA_DIR / "test" / "y_test.txt") - 1).to_numpy()

model = tf.keras.models.load_model(MODELS_DIR / "model.keras")
print("test set:", X_test.shape)

## Évaluation du modèle Keras sur le test

In [ ]:
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

test_accuracy = (y_pred == y_test).mean()
report = classification_report(y_test, y_pred, target_names=label_names, output_dict=True)
macro_f1 = report["macro avg"]["f1-score"]

print(f"accuracy: {test_accuracy:.3f}")
print(f"macro F1: {macro_f1:.3f}")
print()
print(classification_report(y_test, y_pred, target_names=label_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=label_names).plot(ax=ax, xticks_rotation=45, cmap="Greens")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png")
plt.show()

## Analyse des confusions

_À compléter après exécution en regardant la matrice ci-dessus : quelles paires d'activités sont confondues, et pourquoi ça a du sens physiquement._

## Export TensorFlow Lite (float32)

Conversion directe, sans quantification : mêmes poids que le modèle Keras, dans un format plus compact, pensé pour l'inférence sur appareil restreint.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float32 = converter.convert()

path_float32 = MODELS_DIR / "model_float32.tflite"
path_float32.write_bytes(tflite_float32)
print(f"{path_float32.name}: {path_float32.stat().st_size / 1024:.1f} KB")

## Export TensorFlow Lite (int8, quantification complète)

Ici les poids et les activations sont convertis en entiers 8 bits : ça réduit fortement la taille et accélère l'inférence, notamment via des noyaux optimisés comme ESP-NN. Il faut fournir un `representative_dataset` : un échantillon des données d'entraînement déjà normalisées, que le convertisseur utilise pour estimer la plage de valeurs à quantifier.

In [ ]:
def representative_dataset():
    rng = np.random.default_rng(42)
    indices = rng.choice(len(X_fit), size=200, replace=False)
    for i in indices:
        yield [X_fit[i:i+1]]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8
tflite_int8 = converter_int8.convert()

path_int8 = MODELS_DIR / "model_int8.tflite"
path_int8.write_bytes(tflite_int8)
print(f"{path_int8.name}: {path_int8.stat().st_size / 1024:.1f} KB")

## Vérifier les deux versions .tflite sur le test

Le modèle int8 attend des entrées entières : il faut les quantifier nous-même avec le `scale` et le `zero_point` lus dans `get_input_details()` (`valeur_entière = valeur_réelle / scale + zero_point`), puis déquantifier la sortie de la même façon pour retrouver des probabilités comparables.

In [ ]:
def evaluate_tflite(tflite_model, X, y, quantized_io):
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    predictions = np.zeros(len(X), dtype="int64")
    for i in range(len(X)):
        sample = X[i:i+1]
        if quantized_io:
            scale, zero_point = input_details["quantization"]
            sample = np.round(sample / scale + zero_point).astype(input_details["dtype"])
        interpreter.set_tensor(input_details["index"], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details["index"])[0]
        predictions[i] = np.argmax(output)
    accuracy = (predictions == y).mean()
    f1 = classification_report(y, predictions, target_names=label_names, output_dict=True)["macro avg"]["f1-score"]
    return accuracy, f1

accuracy_float32, f1_float32 = evaluate_tflite(tflite_float32, X_test, y_test, quantized_io=False)
accuracy_int8, f1_int8 = evaluate_tflite(tflite_int8, X_test, y_test, quantized_io=True)

print(f"tflite float32 -> accuracy {accuracy_float32:.3f}, macro F1 {f1_float32:.3f}")
print(f"tflite int8    -> accuracy {accuracy_int8:.3f}, macro F1 {f1_int8:.3f}")

## Tableau comparatif et sauvegarde des métriques

In [ ]:
comparison = {
    "keras": {
        "size_kb": (MODELS_DIR / "model.keras").stat().st_size / 1024,
        "accuracy": float(test_accuracy),
        "macro_f1": float(macro_f1),
    },
    "tflite_float32": {
        "size_kb": path_float32.stat().st_size / 1024,
        "accuracy": float(accuracy_float32),
        "macro_f1": float(f1_float32),
    },
    "tflite_int8": {
        "size_kb": path_int8.stat().st_size / 1024,
        "accuracy": float(accuracy_int8),
        "macro_f1": float(f1_int8),
    },
}

with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(comparison, f, indent=2)

pd.DataFrame(comparison).T

## Observations

_À compléter après exécution : analyse des confusions, écart de taille et de précision entre les trois versions du modèle._